# Final Feature Preprocessing

This notebook prepares the final non-agent feature set for close-price modeling. Agent and office features are intentionally excluded because the significance check showed only small effects and high overfitting risk.

## Notes From Edison's Property Notebook

Important choices carried forward:

- Filter to `PropertyType == "Residential"` and `PropertySubType == "SingleFamilyResidence"`.
- Drop invalid/tiny `ClosePrice < 10_000` rows.
- Review/filter extreme `ClosePrice > 20_000_000` rows before modeling.
- Keep the selected structural property features.
- Add `HomeAge = 2026 - YearBuilt` as a derived feature.
- Fill missing boolean amenity flags as `False` because missing usually means the amenity was not reported/present.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


In [ ]:
DATA_FILE = Path("CRMLSSold_combined.csv")
OUTPUT_FILE = Path("preprocessed_final_features.csv")

CURRENT_YEAR = 2026
MIN_CLOSE_PRICE = 10_000
MAX_CLOSE_PRICE = 20_000_000

location_features = [
    "Latitude",
    "Longitude",
    "City",
    "PostalCode",
    "CountyOrParish",
    "MLSAreaMajor",
    "HighSchoolDistrict",
]

property_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "YearBuilt",
    "Stories",
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "GarageSpaces",
    "ParkingTotal",
]

target = "ClosePrice"
filter_cols = ["PropertyType", "PropertySubType"]
selected_cols = filter_cols + location_features + property_features + [target]


## Load Data

In [ ]:
df = pd.read_csv(DATA_FILE, dtype=str, keep_default_na=False, low_memory=False)
df.shape


## Filter To Final Modeling Population

In [ ]:
work = df[selected_cols].copy()
raw_rows = len(work)

work = work[
    (work["PropertyType"] == "Residential")
    & (work["PropertySubType"] == "SingleFamilyResidence")
].copy()
after_sfr_filter = len(work)

numeric_cols = [
    "Latitude",
    "Longitude",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "YearBuilt",
    "Stories",
    "GarageSpaces",
    "ParkingTotal",
    target,
]

for col in numeric_cols:
    work[col] = pd.to_numeric(work[col].replace("", np.nan), errors="coerce")

work = work[(work[target] >= MIN_CLOSE_PRICE) & (work[target] <= MAX_CLOSE_PRICE)].copy()
after_target_filter = len(work)

work = work.dropna(subset=["Latitude", "Longitude"]).copy()
after_coordinate_filter = len(work)

print(f"Raw selected rows: {raw_rows:,}")
print(f"After Residential + SingleFamilyResidence: {after_sfr_filter:,}")
print(f"After ${MIN_CLOSE_PRICE:,} <= ClosePrice <= ${MAX_CLOSE_PRICE:,}: {after_target_filter:,}")
print(f"After dropping missing latitude/longitude: {after_coordinate_filter:,}")


## Clean Feature Types

In [ ]:
bool_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN"]

for col in bool_cols:
    normalized = work[col].astype(str).str.strip().str.lower()
    work[col] = (
        normalized
        .map({"true": 1, "false": 0, "1": 1, "0": 0, "yes": 1, "no": 0})
        .fillna(0)
        .astype(int)
    )

categorical_cols = ["City", "PostalCode", "CountyOrParish", "MLSAreaMajor", "HighSchoolDistrict"]

for col in categorical_cols:
    work[col] = work[col].astype(str).str.strip().replace("", "Unknown")

work["PostalCode"] = work["PostalCode"].str.extract(r"(\d{5})", expand=False).fillna("Unknown")

work["HomeAge"] = CURRENT_YEAR - work["YearBuilt"]
work.loc[(work["HomeAge"] < 0) | (work["HomeAge"] > 250), "HomeAge"] = np.nan


## Impute Numeric Missing Values

Use medians for simple baseline preprocessing. A production modeling pipeline can replace this with train-only imputation inside scikit-learn.

In [ ]:
impute_cols = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "YearBuilt",
    "Stories",
    "GarageSpaces",
    "ParkingTotal",
    "HomeAge",
]

impute_values = {}

for col in impute_cols:
    impute_values[col] = work[col].median()
    work[col] = work[col].fillna(impute_values[col])

pd.Series(impute_values, name="median_impute_value")


## Write Final Dataset

In [ ]:
final_cols = location_features + property_features + ["HomeAge", target]
final_df = work[final_cols].copy()

final_df.to_csv(OUTPUT_FILE, index=False)

print(f"Final rows: {len(final_df):,}")
print(f"Final columns: {len(final_df.columns):,}")
print(f"Remaining missing values: {final_df.isna().sum().sum():,}")
print(f"Wrote {OUTPUT_FILE}")

final_df.head()


In [ ]:
final_df.dtypes
